# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
# Same preprocessing as the previous lab
spaceship = spaceship.dropna().reset_index(drop=True)
print("Shape after dropping NaN:", spaceship.shape)

# Cabin -> deck letter only (A-T)
spaceship["Cabin"] = spaceship["Cabin"].str[0]

# Drop identifiers (no predictive value)
spaceship = spaceship.drop(columns=["PassengerId", "Name"])

# Target and features
y = spaceship["Transported"].astype(int)
X = pd.get_dummies(spaceship.drop(columns="Transported"), drop_first=True)
X = X.astype(float)
print("Features:", X.shape[1])
X.head()

Shape after dropping NaN: (6606, 14)
Features: 19


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,24.0,109.0,9.0,25.0,549.0,44.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,58.0,43.0,3576.0,0.0,6715.0,49.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3,33.0,0.0,1283.0,371.0,3329.0,193.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,16.0,303.0,70.0,151.0,565.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [4]:
# Feature scaling on the numerical columns (fitted on the train set below to avoid leakage)
from sklearn.preprocessing import StandardScaler
num_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

**Perform Train Test Split**

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(X_train.shape, X_test.shape)

(5284, 19) (1322, 19)


**Baseline** - KNN from the previous lab, to compare fairly

In [6]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

results = {}
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
results["KNN (baseline)"] = (accuracy_score(y_train, knn.predict(X_train)), accuracy_score(y_test, knn.predict(X_test)))
print(results["KNN (baseline)"])

(0.8344057532172596, 0.7708018154311649)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [7]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# Bagging: sampling WITH replacement (bootstrap=True)
bagging = BaggingClassifier(DecisionTreeClassifier(), n_estimators=200, max_samples=0.8,
                            bootstrap=True, n_jobs=-1, random_state=42).fit(X_train, y_train)
results["Bagging"] = (accuracy_score(y_train, bagging.predict(X_train)), accuracy_score(y_test, bagging.predict(X_test)))

# Pasting: sampling WITHOUT replacement (bootstrap=False)
pasting = BaggingClassifier(DecisionTreeClassifier(), n_estimators=200, max_samples=0.8,
                            bootstrap=False, n_jobs=-1, random_state=42).fit(X_train, y_train)
results["Pasting"] = (accuracy_score(y_train, pasting.predict(X_train)), accuracy_score(y_test, pasting.predict(X_test)))
print("Bagging :", results["Bagging"])
print("Pasting :", results["Pasting"])

Bagging : (0.9401968205904617, 0.7844175491679274)
Pasting : (0.9407645722937169, 0.7738275340393344)


- Random Forests

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=5,
                            n_jobs=-1, random_state=42).fit(X_train, y_train)
results["Random Forest"] = (accuracy_score(y_train, rf.predict(X_train)), accuracy_score(y_test, rf.predict(X_test)))
print(results["Random Forest"])

# Most important features
pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(8)

(0.836109008327025, 0.7980332829046899)


CryoSleep_True       0.183720
Spa                  0.148015
VRDeck               0.142654
RoomService          0.124032
FoodCourt            0.097140
ShoppingMall         0.088264
Age                  0.065704
HomePlanet_Europa    0.036285
dtype: float64

- Gradient Boosting

In [9]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42).fit(X_train, y_train)
results["Gradient Boosting"] = (accuracy_score(y_train, gb.predict(X_train)), accuracy_score(y_test, gb.predict(X_test)))
print(results["Gradient Boosting"])

(0.8332702498107495, 0.7791225416036308)


- Adaptive Boosting

In [10]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=2), n_estimators=200, learning_rate=0.5, random_state=42).fit(X_train, y_train)
results["AdaBoost"] = (accuracy_score(y_train, ada.predict(X_train)), accuracy_score(y_test, ada.predict(X_test)))
print(results["AdaBoost"])

(0.7990158970476912, 0.783661119515885)


**Comparison of all models**

In [11]:
comparison = pd.DataFrame(results, index=["Train accuracy", "Test accuracy"]).T
comparison["Gap (overfit)"] = comparison["Train accuracy"] - comparison["Test accuracy"]
comparison.sort_values("Test accuracy", ascending=False).round(4)

,Train accuracy,Test accuracy,Gap (overfit)
Random Forest,0.8361,0.7980,0.0381
Bagging,0.9402,0.7844,0.1558
AdaBoost,0.7990,0.7837,0.0154
Gradient Boosting,0.8333,0.7791,0.0541
Pasting,0.9408,0.7738,0.1669
KNN (baseline),0.8344,0.7708,0.0636


Which model is the best and why?

In [12]:
# Conclusion
# Random Forest is the best model: highest test accuracy (~0.798 vs ~0.771 for the KNN baseline)
# with a moderate train/test gap (~0.04), so it generalises well.
# - Bagging and Pasting reach ~0.94 on train but only ~0.77-0.78 on test: fully grown trees overfit heavily.
# - Random Forest adds feature randomness on top of bagging and we limited depth (max_depth=10,
#   min_samples_leaf=5), which decorrelates the trees and reduces variance.
# - Gradient Boosting and AdaBoost are more stable (small gap, AdaBoost the smallest) but slightly
#   less accurate here; with more tuning (learning_rate, n_estimators) boosting could catch up.
# - All ensembles beat or match the KNN baseline; differences between them are small (~1-2 points),
#   so a cross-validation / GridSearchCV would be needed to confirm the ranking.
